# Quick Hyperparameter Tuning (트리 모델 5개)

전처리 후 LightGBM / XGBoost / CatBoost / RandomForest / ExtraTrees Optuna HPO

**목적**: 전처리 파이프라인 위에 "모델의 최적" 파라미터를 확보하여 이후 피처 실험의 기준선을 확립하려 함

**특징**:
- LightGBM / XGBoost / CatBoost: n_estimators=10000 고정 + early_stopping
- RandomForest / ExtraTrees: n_estimators 탐색 (100~1000)
- Colab GPU 자동 감지 → LightGBM/XGBoost/CatBoost GPU 가속
- 최적 파라미터 + 학습된 모델 자동 저장 (Local/Colab 모두 읽기)

## 1. 환경 설정 및 데이터 로드

In [1]:
import os, sys

try:
    import google.colab
    if not os.path.exists("/content/project/setup.py"):
        os.system("pip install -q gdown")
        os.system("gdown --id 1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I -O /content/code.zip")
        os.system("unzip -qo /content/code.zip -d /content/project")
        os.makedirs("/content/project/0_data", exist_ok=True)
        os.system("gdown --id 1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO -O /content/project/0_data/dataset.zip")
        os.system("unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data")
        os.remove("/content/project/0_data/dataset.zip")
    if not os.path.exists("/content/project/2_preprocessing/cleaning.py"):
        os.system("gdown --id 1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr -O /content/preprocessing.zip")
        os.system("unzip -qo /content/preprocessing.zip -d /content/project")
    sys.path.insert(0, "/content/project")
    %run /content/project/setup.py
except ImportError:
    %run ../setup.py

import numpy as np
import pandas as pd
import optuna
import joblib
import json as json_mod
import warnings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# GPU 감지
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
    GPU_NAME = torch.cuda.get_device_name(0) if HAS_GPU else "N/A"
except ImportError:
    HAS_GPU = False
    GPU_NAME = "N/A"

print(f"GPU: {HAS_GPU}" + (f" ({GPU_NAME})" if HAS_GPU else ""))

# 트리 모델
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor

# 프로젝트 utils
from sklearn.preprocessing import RobustScaler
from utils.config import PROJECT_ROOT, SEED, TARGET_COL, ENV, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs
from utils.aggregate import merge_with_target
from utils.evaluate import evaluate, postprocess, rmse

# 전처리 모듈
sys.path.insert(0, os.path.join(PROJECT_ROOT, "2_preprocessing"))
from cleaning import run_cleaning
from outlier import run_outlier_treatment
from aggregation import run_aggregation

# ─── 저장 경로 설정 (Local / Colab 분기) ───
HPO_DIR = os.path.join(OUTPUT_DIR, "hpo")
os.makedirs(HPO_DIR, exist_ok=True)
print(f"ENV: {ENV}")
print(f"저장 경로: {HPO_DIR}")

# 데이터 로드
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)

패키지 설치 중: ['optuna', 'boruta', 'pytorch-tabnet', 'rtdl-revisiting-models']
setup 완료
GPU: False
ENV: colab
저장 경로: /content/project/4_output/hpo
Xs: (174980, 1091)  |  Ys: train=26,247, val=8,749, test=8,749


## 2. 전처리 파이프라인 (클리닝 → 이상치 → 집계)

In [2]:
# --- Step 1: 클리닝 (상수/고결측/중복 제거 + 결측 imputation) ---
xs_train, xs_val, xs_test, clean_cols, _ = run_cleaning(
    xs, feat_cols, xs_dict,
    const_threshold=1e-6,
    missing_threshold=0.5,
    remove_duplicates=True,
)

# --- Step 2: 이상치 처리 (Winsorization) ---
xs_train, xs_val, xs_test, _ = run_outlier_treatment(
    xs_train, xs_val, xs_test, clean_cols,
    lower_pct=0.01, upper_pct=0.99,
)

# --- Step 3: Die → Unit 집계 ---
unit_train, unit_val, unit_test, unit_feat_cols = run_aggregation(
    xs_train, xs_val, xs_test, clean_cols,
    agg_funcs=["mean", "std", "min", "max"],
    use_position_pivot=False,
    save_csv=False,
)

클리닝 파이프라인 시작
원본 feature 수: 1087
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 982개
    컬럼: 1087 → 982 (105개 제거)
    DataFrame: (104988, 986)

[고결측 제거] threshold=50%
  제거: 5개, 잔여: 977개
    컬럼: 982 → 977 (5개 제거)
    DataFrame: (104988, 981)

[중복 컬럼 제거] sample_n=5000
  제거: 26개, 잔여: 951개
    컬럼: 977 → 951 (26개 제거)
    DataFrame: (104988, 955)

[결측 imputation] train median 기준
  imputation 후 잔여 결측: 0

클리닝 완료: 1087 → 951 features (136개 제거)
  train: (104988, 955)
  val:   (34996, 955)
  test:  (34996, 955)
이상치 처리 파이프라인 시작
[이상치 탐지] IQR × 1.5
  이상치 > 5%: 165개
  이상치 > 10%: 63개
[Winsorization] lower=1%, upper=99%
  적용 feature: 951개
[이상치 탐지] IQR × 1.5
  이상치 > 5%: 164개
  이상치 > 10%: 63개

이상치 처리 완료
  이상치 >5%  feature: 165 → 164 (1개 감소)
  이상치 >10% feature: 63 → 63 (0개 감소)
  train: (104988, 955)
Die → Unit 집계 시작
  agg_funcs: ['mean', 'std', 'min', 'max']
  position_pivot: False
집계 완료: 26,247 units × 3,804 features (agg: ['mean', 'std', 'min', 'max'])
집계 완료: 8,749 units × 3,804 features (agg: ['mean', 'st

## 3. 학습 데이터 준비

In [3]:
# Target merge
X_train, y_train = merge_with_target(unit_train, split="train")
X_val, y_val = merge_with_target(unit_val, split="validation")

assert X_train.isnull().sum().sum() == 0, "X_train에 NaN 존재!"
assert X_val.isnull().sum().sum() == 0, "X_val에 NaN 존재!"

# RobustScaler
scaler = RobustScaler()
X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_val_s = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns, index=X_val.index)

print(f"\n학습: {X_train_s.shape}, 검증: {X_val_s.shape}")
print(f"y_train: mean={y_train.mean():.6f}, zero={(y_train==0).mean()*100:.1f}%")
print(f"y_val:   mean={y_val.mean():.6f}, zero={(y_val==0).mean()*100:.1f}%")

# 단순 baseline (비교 기준선)
print(f"\n--- 단순 Baseline ---")
evaluate(y_val, np.zeros(len(y_val)), label="all-zero")
evaluate(y_val, np.full(len(y_val), y_train.mean()), label="train-mean")

Merge (train): X=(26247, 3804), y=(26247,), y_mean=0.002515
Merge (validation): X=(8749, 3804), y=(8749,), y_mean=0.002505

학습: (26247, 3804), 검증: (8749, 3804)
y_train: mean=0.002515, zero=70.8%
y_val:   mean=0.002505, zero=70.8%

--- 단순 Baseline ---
[all-zero] RMSE = 0.006331  (n=8,749, zero=6,194(70.8%))
[train-mean] RMSE = 0.005814  (n=8,749, zero=6,194(70.8%))


0.00581430473855135

## 4. Quick HPO

| 모델 | trials | n_estimators | GPU |
|------|--------|-------------|-----|
| LightGBM | 500 | 10000 (early_stop) | O |
| XGBoost | 500 | 10000 (early_stop) | O |
| CatBoost | 500 | 10000 (early_stop) | O |
| RandomForest | 100 | 100~1000 탐색 | X |
| ExtraTrees | 100 | 100~1000 탐색 | X |

In [4]:
# ─── HPO 공통 설정 (야간용: 세게) ───
N_TRIALS_BOOST = 500  # LightGBM, XGBoost, CatBoost
N_TRIALS_TREE = 100   # RF, ExtraTrees
hpo_results = {}      # 모델별 결과 저장

print(f"Quick HPO 설정 (야간용)")
print(f"  Boost 모델: {N_TRIALS_BOOST} trials")
print(f"  Tree 모델:  {N_TRIALS_TREE} trials")
print(f"  평가: Val RMSE (train→val 단일 split)")
print(f"  GPU: {HAS_GPU}")
print(f"  저장: {HPO_DIR}")

Quick HPO 설정 (야간용)
  Boost 모델: 500 trials
  Tree 모델:  100 trials
  평가: Val RMSE (train→val 단일 split)
  GPU: False
  저장: /content/project/4_output/hpo


### 4-1. LightGBM

In [5]:
# Optuna objective: 1 trial = LightGBM 1회 학습 → Val RMSE 반환
# n_estimators=10000 고정, early_stopping이 실제 반복 수 결정 (보통 200~500에서 멈춤)
def lgb_objective(trial):
    params = {
        "n_estimators": 10000,                                                      # 고정 (early_stopping이 끊음)
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 128),                     # 26K행 → 128 이상은 과적합
        "max_depth": trial.suggest_int("max_depth", -1, 12),                        # -1=무제한, num_leaves가 주 제어
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),       # y=0이 70%라 넉넉히
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "subsample_freq": 1,                                                        # subsample 활성화에 필요
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.2, 0.8),      # 3804열 → 낮게
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-6, 5.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-6, 5.0, log=True),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 0.01),         # deeper split gain≈0.001~0.01이라 0.01이 상한
        "path_smooth": trial.suggest_float("path_smooth", 0.0, 10.0),               # leaf 예측 평활화 (zero-inflated 효과적)
        "random_state": SEED,
        "n_jobs": -1,
        "verbose": -1,
        "device": "gpu" if HAS_GPU else "cpu",
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_train_s, y_train,
        eval_set=[(X_val_s, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)],
    )
    pred = postprocess(model.predict(X_val_s))
    return rmse(y_val, pred)

print("=" * 50)
print("[LightGBM] Optuna HPO 시작")
lgb_study = optuna.create_study(direction="minimize", study_name="lgb_quick")
lgb_study.optimize(lgb_objective, n_trials=N_TRIALS_BOOST, show_progress_bar=True)

print(f"\n  Best RMSE: {lgb_study.best_value:.6f}")
print(f"  Best params:")
for k, v in lgb_study.best_params.items():
    print(f"    {k}: {v}")

hpo_results["LightGBM"] = {
    "best_rmse": lgb_study.best_value,
    "best_params": lgb_study.best_params,
    "study": lgb_study,
}

[LightGBM] Optuna HPO 시작


  0%|          | 0/500 [00:00<?, ?it/s]


  Best RMSE: 0.005756
  Best params:
    learning_rate: 0.012159085193696212
    num_leaves: 103
    max_depth: 10
    min_child_samples: 145
    subsample: 0.6784084803658785
    colsample_bytree: 0.25420811555273726
    reg_alpha: 0.05219420305343204
    reg_lambda: 0.9744061750035492
    min_split_gain: 4.9338160959470715e-06
    path_smooth: 6.83169390979537


### 4-2. XGBoost

In [6]:
# XGBoost: early_stopping으로 n_estimators 자동 결정
# max_delta_step: zero-inflated 전용, leaf weight 극단값 억제 (공식 문서 권장)
def xgb_objective(trial):
    params = {
        "n_estimators": 10000,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 5, 100),          # lgb의 min_child_samples 역할
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.2, 0.8),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.3, 1.0),    # 레벨별 피처 샘플링
        "gamma": trial.suggest_float("gamma", 0, 0.5),                              # y가 작아서 gain도 작음 → 낮게
        "max_delta_step": trial.suggest_float("max_delta_step", 0, 1.0),            # leaf weight≈0.01 수준이라 0~1.0 범위
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-6, 5.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-6, 5.0, log=True),
        "random_state": SEED,
        "n_jobs": -1,
        "verbosity": 0,
        "early_stopping_rounds": 50,
        "device": "cuda" if HAS_GPU else "cpu",
    }
    model = xgb.XGBRegressor(**params)
    model.fit(X_train_s, y_train, eval_set=[(X_val_s, y_val)], verbose=False)
    pred = postprocess(model.predict(X_val_s))
    return rmse(y_val, pred)

print("=" * 50)
print("[XGBoost] Optuna HPO 시작")
xgb_study = optuna.create_study(direction="minimize", study_name="xgb_quick")
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS_BOOST, show_progress_bar=True)

print(f"\n  Best RMSE: {xgb_study.best_value:.6f}")
print(f"  Best params:")
for k, v in xgb_study.best_params.items():
    print(f"    {k}: {v}")

hpo_results["XGBoost"] = {
    "best_rmse": xgb_study.best_value,
    "best_params": xgb_study.best_params,
    "study": xgb_study,
}

[XGBoost] Optuna HPO 시작


  0%|          | 0/500 [00:00<?, ?it/s]


  Best RMSE: 0.005776
  Best params:
    learning_rate: 0.08409445663091217
    max_depth: 7
    min_child_weight: 47
    subsample: 0.8497935301590033
    colsample_bytree: 0.798760210707177
    colsample_bylevel: 0.9163567069311098
    gamma: 0.00027391190828931057
    max_delta_step: 0.7970704471937062
    reg_alpha: 0.2139447841682324
    reg_lambda: 0.040075051482258574


### 4-3. RandomForest

In [5]:
# RF는 early_stopping 없음 → n_estimators도 탐색 대상 (step=100)
# GPU 미지원이라 Boost보다 느림, trials를 적게 잡음
def rf_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=100),
        "max_depth": trial.suggest_int("max_depth", 10, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 50),     # 상한 20→50 (26K행 안정성)
        "max_features": trial.suggest_float("max_features", 0.02, 0.5),       # 하한 0.02 (sqrt(3804)=0.016 포함)
        "max_samples": trial.suggest_float("max_samples", 0.5, 1.0),          # bootstrap 비율 → 다양성 증가
        "ccp_alpha": trial.suggest_float("ccp_alpha", 1e-8, 1e-4, log=True),  # 사후 가지치기 (0.001이면 트리 통째로 잘림)
        "random_state": SEED,
        "n_jobs": -1,
    }
    model = RandomForestRegressor(**params)
    model.fit(X_train_s, y_train)
    pred = postprocess(model.predict(X_val_s))
    return rmse(y_val, pred)

print("=" * 50)
print("[RandomForest] Optuna HPO 시작")
rf_study = optuna.create_study(direction="minimize", study_name="rf_quick")
rf_study.optimize(rf_objective, n_trials=N_TRIALS_TREE, show_progress_bar=True)

print(f"\n  Best RMSE: {rf_study.best_value:.6f}")
print(f"  Best params:")
for k, v in rf_study.best_params.items():
    print(f"    {k}: {v}")

hpo_results["RandomForest"] = {
    "best_rmse": rf_study.best_value,
    "best_params": rf_study.best_params,
    "study": rf_study,
}

[RandomForest] Optuna HPO 시작


  0%|          | 0/100 [00:00<?, ?it/s]


  Best RMSE: 0.005759
  Best params:
    n_estimators: 200
    max_depth: 28
    min_samples_split: 2
    min_samples_leaf: 49
    max_features: 0.12638575533467306
    max_samples: 0.6997955981490869
    ccp_alpha: 1.014400770989981e-08


### 4-4. ExtraTrees

In [ ]:
# ExtraTrees: RF와 구조 동일, 분할 기준이 랜덤이라 RF보다 빠르고 분산 낮음
def et_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=100),
        "max_depth": trial.suggest_int("max_depth", 10, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 50),
        "max_features": trial.suggest_float("max_features", 0.02, 0.5),
        "ccp_alpha": trial.suggest_float("ccp_alpha", 1e-8, 1e-4, log=True),
        "random_state": SEED,
        "n_jobs": -1,
    }
    model = ExtraTreesRegressor(**params)
    model.fit(X_train_s, y_train)
    pred = postprocess(model.predict(X_val_s))
    return rmse(y_val, pred)

print("=" * 50)
print("[ExtraTrees] Optuna HPO 시작")
et_study = optuna.create_study(direction="minimize", study_name="et_quick")
et_study.optimize(et_objective, n_trials=N_TRIALS_TREE, show_progress_bar=True)

print(f"\n  Best RMSE: {et_study.best_value:.6f}")
print(f"  Best params:")
for k, v in et_study.best_params.items():
    print(f"    {k}: {v}")

hpo_results["ExtraTrees"] = {
    "best_rmse": et_study.best_value,
    "best_params": et_study.best_params,
    "study": et_study,
}

### 4-5. CatBoost

In [ ]:
# CatBoost: early_stopping으로 n_estimators 자동 결정
# Ordered boosting + symmetric trees → 기본적으로 과적합에 강함
def cb_objective(trial):
    params = {
        "iterations": 10000,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 20, 200),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.2, 0.8),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "random_seed": SEED,
        "verbose": 0,
        "early_stopping_rounds": 50,
        "task_type": "GPU" if HAS_GPU else "CPU",
    }
    model = CatBoostRegressor(**params)
    model.fit(X_train_s, y_train, eval_set=(X_val_s, y_val))
    pred = postprocess(model.predict(X_val_s))
    return rmse(y_val, pred)

print("=" * 50)
print("[CatBoost] Optuna HPO 시작")
cb_study = optuna.create_study(direction="minimize", study_name="cb_quick")
cb_study.optimize(cb_objective, n_trials=N_TRIALS_BOOST, show_progress_bar=True)

print(f"\n  Best RMSE: {cb_study.best_value:.6f}")
print(f"  Best params:")
for k, v in cb_study.best_params.items():
    print(f"    {k}: {v}")

hpo_results["CatBoost"] = {
    "best_rmse": cb_study.best_value,
    "best_params": cb_study.best_params,
    "study": cb_study,
}

## 5. 결과 비교

In [ ]:
# ─── 5개 모델 Best RMSE 비교표 ───
comparison = pd.DataFrame([
    {"Model": name, "Best Val RMSE": r["best_rmse"]}
    for name, r in hpo_results.items()
]).sort_values("Best Val RMSE").reset_index(drop=True)
comparison.index += 1

print("=" * 60)
print("Quick HPO 결과 비교")
print("=" * 60)
print(comparison.to_string())

# ─── 모델별 최적 파라미터 출력 (복사용) ───
print("\n" + "=" * 60)
print("최적 파라미터 (복사하여 사용)")
print("=" * 60)
for name, r in hpo_results.items():
    print(f"\n# --- {name} ---")
    print(f"{name.lower().replace(' ', '_')}_best_params = {{")
    items = list(r["best_params"].items())
    for i, (k, v) in enumerate(items):
        comma = "," if i < len(items) - 1 else ""
        if isinstance(v, float):
            print(f'    "{k}": {v:.6f}{comma}')
        else:
            print(f'    "{k}": {v}{comma}')
    print("}")
    print(f'# Val RMSE: {r["best_rmse"]:.6f}')

## 6. 모델 & 파라미터 저장

최적 파라미터로 재학습 후 저장:
- `best_params.json`: 5개 모델 최적 파라미터 (다음 노트북에서 로드)
- `{모델명}_best.joblib`: 학습된 모델 가중치
- `{모델명}_study.pkl`: Optuna study 객체 (나중에 이어서 탐색 가능)
- Colab: 자동으로 브라우저 다운로드

In [ ]:
import pickle

# ─── Best 파라미터로 5개 모델 재학습 ───
# HPO에서 찾은 최적 파라미터 + 고정 파라미터를 합쳐서 전체 train으로 학습
# 저장된 모델은 이후 피처 실험 시 바로 로드하여 사용 가능
best_models = {}

# LightGBM
lgb_params = {**lgb_study.best_params, "n_estimators": 10000, "random_state": SEED,
              "n_jobs": -1, "verbose": -1, "device": "gpu" if HAS_GPU else "cpu"}
lgb_best = lgb.LGBMRegressor(**lgb_params)
lgb_best.fit(X_train_s, y_train, eval_set=[(X_val_s, y_val)],
             callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
best_models["LightGBM"] = lgb_best

# XGBoost
xgb_params = {**xgb_study.best_params, "n_estimators": 10000, "random_state": SEED,
              "n_jobs": -1, "verbosity": 0, "early_stopping_rounds": 50,
              "device": "cuda" if HAS_GPU else "cpu"}
xgb_best = xgb.XGBRegressor(**xgb_params)
xgb_best.fit(X_train_s, y_train, eval_set=[(X_val_s, y_val)], verbose=False)
best_models["XGBoost"] = xgb_best

# RandomForest
rf_params = {**rf_study.best_params, "random_state": SEED, "n_jobs": -1}
rf_best = RandomForestRegressor(**rf_params)
rf_best.fit(X_train_s, y_train)
best_models["RandomForest"] = rf_best

# ExtraTrees
et_params = {**et_study.best_params, "random_state": SEED, "n_jobs": -1}
et_best = ExtraTreesRegressor(**et_params)
et_best.fit(X_train_s, y_train)
best_models["ExtraTrees"] = et_best
# CatBoost
cb_params = {**cb_study.best_params, "iterations": 10000, "random_seed": SEED,
             "verbose": 0, "early_stopping_rounds": 50,
             "task_type": "GPU" if HAS_GPU else "CPU"}
cb_best = CatBoostRegressor(**cb_params)
cb_best.fit(X_train_s, y_train, eval_set=(X_val_s, y_val))
best_models["CatBoost"] = cb_best

# ─── Val RMSE 재확인 ───
print("=" * 60)
print("Best 모델 Val RMSE 재확인")
print("=" * 60)
for name, model in best_models.items():
    pred = postprocess(model.predict(X_val_s))
    evaluate(y_val, pred, label=f"{name} (best)")

# ─── 저장 ───
print(f"\n저장 중... → {HPO_DIR}")

# 1. 최적 파라미터 JSON (다음 노트북에서 json.load로 바로 사용)
params_to_save = {}
for name, r in hpo_results.items():
    params_to_save[name] = {
        "best_rmse": r["best_rmse"],
        "best_params": r["best_params"],
    }
with open(os.path.join(HPO_DIR, "best_params.json"), "w") as f:
    json_mod.dump(params_to_save, f, indent=2, ensure_ascii=False)
print("  best_params.json 저장 완료")

# 2. 학습된 모델 저장 (joblib → joblib.load로 복원)
for name, model in best_models.items():
    safe_name = name.lower().replace(" ", "_")
    fname = f"{safe_name}_best.joblib"
    joblib.dump(model, os.path.join(HPO_DIR, fname))
    print(f"  {fname} 저장 완료")

# 3. Optuna study 저장 (pickle → 나중에 이어서 탐색 가능)
studies_dict = {"LightGBM": lgb_study, "XGBoost": xgb_study, "CatBoost": cb_study,
                "RandomForest": rf_study, "ExtraTrees": et_study}
for name, study in studies_dict.items():
    safe_name = name.lower().replace(" ", "_")
    fname = f"{safe_name}_study.pkl"
    with open(os.path.join(HPO_DIR, fname), "wb") as f:
        pickle.dump(study, f)
    print(f"  {fname} 저장 완료")

# 4. Scaler 저장 (피처 실험 시 동일 scaler 재사용)
joblib.dump(scaler, os.path.join(HPO_DIR, "scaler.joblib"))
print("  scaler.joblib 저장 완료")

print(f"\n모든 파일 저장 완료 → {HPO_DIR}")

# ─── Colab: 브라우저 다운로드 ───
if ENV == "colab":
    from google.colab import files as colab_files
    colab_files.download(os.path.join(HPO_DIR, "best_params.json"))
    print("\nColab: best_params.json 브라우저 다운로드 실행")
    print("나머지 모델 파일은 아래 셀에서 선택적으로 다운로드하세요")

In [ ]:
# ─── Colab 전용: 모델 파일 선택적 다운로드 ───
# (로컬에서는 이미 4_output/hpo/ 에 저장되어 있으므로 스킵)
if ENV == "colab":
    from google.colab import files as colab_files
    for fname in os.listdir(HPO_DIR):
        fpath = os.path.join(HPO_DIR, fname)
        size_mb = os.path.getsize(fpath) / 1024 / 1024
        print(f"  {fname} ({size_mb:.1f} MB)")
    print("\n다운로드할 파일을 개별 실행하세요:")
    print('  colab_files.download(os.path.join(HPO_DIR, "lightgbm_best.joblib"))')
else:
    print(f"로컬 저장 완료: {HPO_DIR}")
    for fname in os.listdir(HPO_DIR):
        fpath = os.path.join(HPO_DIR, fname)
        size_mb = os.path.getsize(fpath) / 1024 / 1024
        print(f"  {fname} ({size_mb:.1f} MB)")

## 7. Optuna 최적화 과정 시각화

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 2, figsize=(16, 14))
studies = [
    ("LightGBM", lgb_study),
    ("XGBoost", xgb_study),
    ("CatBoost", cb_study),
    ("RandomForest", rf_study),
    ("ExtraTrees", et_study),
]

for ax, (name, study) in zip(axes.flatten(), studies):
    trials = study.trials
    values = [t.value for t in trials]
    best_so_far = [min(values[:i+1]) for i in range(len(values))]
    ax.plot(values, "o", alpha=0.4, markersize=3, label="trial")
    ax.plot(best_so_far, "-", color="red", linewidth=2, label="best")
    ax.set_title(f"{name} (best: {study.best_value:.6f})")
    ax.set_xlabel("Trial")
    ax.set_ylabel("Val RMSE")
    ax.legend()

axes[-1, -1].set_visible(False)  # 6번째 빈 칸 숨김

plt.suptitle("Quick HPO 최적화 과정", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 8. 파라미터 중요도

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 14))

for ax, (name, study) in zip(axes.flatten(), studies):
    importances = optuna.importance.get_param_importances(study)
    params = list(importances.keys())
    values = list(importances.values())
    ax.barh(params, values, color="steelblue")
    ax.invert_yaxis()
    ax.set_title(f"{name}")
    ax.set_xlabel("Importance")

axes[-1, -1].set_visible(False)  # 6번째 빈 칸 숨김

plt.suptitle("파라미터 중요도 (Optuna fANOVA)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()